# Starbucks LFS Model — Quick Start Demo

This notebook demonstrates how to load and use the **Location Fitness Score (LFS) predictive model** published on [Kaggle Models](https://www.kaggle.com/models/shiratoriseto/starbucks-location-fitness-model).

The model predicts how well a Starbucks location is positioned relative to demand (transit, foot traffic) and competitive supply in Manhattan.

## Step 1 — Load the Model

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# ── model path resolution ────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    _candidates = [
        Path("/kaggle/input/models/shiratoriseto/starbucks-location-fitness-model/other/default/1/lfs_model.joblib"),
        Path("/kaggle/input/starbucks-location-fitness-model/other/default/1/lfs_model.joblib"),
        Path("/kaggle/input/starbucks-location-fitness-model/lfs_model.joblib"),
    ]
    MODEL_PATH = None
    for _p in _candidates:
        if _p.exists():
            MODEL_PATH = _p
            break
    if MODEL_PATH is None:
        import os
        avail = []
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(".joblib"):
                    avail.append(os.path.join(root, f))
        raise FileNotFoundError(f"Model not found. Available .joblib files: {avail}")
else:
    MODEL_PATH = Path("lfs_model.joblib")

model = joblib.load(MODEL_PATH)
print(f"Model loaded: {type(model).__name__}")
print(f"Number of features: {model.n_features_in_}")

## Step 2 — Load the Dataset

In [ ]:
# ── data path resolution ────────────────────────────────────────────
if ON_KAGGLE:
    _candidates = [
        Path("/kaggle/input/manhattan-cafe-wars"),
        Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
    ]
    DATA_DIR = None
    for _p in _candidates:
        if _p.exists():
            DATA_DIR = _p
            break
    if DATA_DIR is None:
        import os
        avail = os.listdir("/kaggle/input") if Path("/kaggle/input").exists() else []
        raise FileNotFoundError(f"Dataset not found. Available: {avail}")
else:
    DATA_DIR = Path("../../dataset-upload/v3")

df = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} cols")
df.head(3)

## Step 3 — Prepare Features & Predict

In [ ]:
# ── feature columns (same order as training) ─────────────────────────
feature_cols = [
    "pluto_numfloors", "pluto_yearbuilt", "pluto_retailarea",
    "pluto_comarea", "pluto_lotarea", "pluto_assesstot",
    "mta_dist_m", "mta_avg_daily_ridership",
    "n_starbucks_500m", "n_dunkin_500m", "n_other_cafe_500m",
    "nearest_competitor_dist_m", "nearest_starbucks_dist_m",
    "tract_population", "tract_median_income",
    "tract_pct_walk_commute", "tract_pct_bachelors_plus",
    "ped_count_nearest",
    "lat", "lon",
]

TARGET = "location_fitness_score"

# Drop NaN rows
df_valid = df[feature_cols + [TARGET]].dropna().copy()
X = df_valid[feature_cols].values
y_actual = df_valid[TARGET].values

# Predict
y_pred = model.predict(X)
df_valid["predicted_lfs"] = y_pred
df_valid["residual"] = y_actual - y_pred

print(f"Predictions made for {len(df_valid)} stores")
print(f"\nActual LFS  — mean: {y_actual.mean():.3f}, std: {y_actual.std():.3f}")
print(f"Predicted   — mean: {y_pred.mean():.3f}, std: {y_pred.std():.3f}")

## Step 4 — Visualize Results

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

mae = mean_absolute_error(y_actual, y_pred)
r2 = r2_score(y_actual, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
ax = axes[0]
ax.scatter(y_actual, y_pred, alpha=0.6, edgecolors="k", linewidth=0.5,
           color="#00704A", s=50)
lo = min(y_actual.min(), y_pred.min()) - 0.1
hi = max(y_actual.max(), y_pred.max()) + 0.1
ax.plot([lo, hi], [lo, hi], "--", color="grey", linewidth=1)
ax.set_xlabel("Actual LFS")
ax.set_ylabel("Predicted LFS")
ax.set_title(f"Actual vs Predicted  (R² = {r2:.3f})")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect("equal")

# Top / Bottom 5 stores
ax = axes[1]
top5 = df_valid.nlargest(5, "predicted_lfs")[["predicted_lfs", TARGET]]
bot5 = df_valid.nsmallest(5, "predicted_lfs")[["predicted_lfs", TARGET]]
combined = pd.concat([top5, bot5])
idx = range(len(combined))
ax.barh(idx, combined["predicted_lfs"], color="#00704A", alpha=0.7, label="Predicted")
ax.barh(idx, combined[TARGET], color="#1E3932", alpha=0.4, label="Actual")
ax.set_yticks(idx)
ax.set_yticklabels([f"Store {i}" for i in combined.index], fontsize=8)
ax.set_xlabel("Location Fitness Score")
ax.set_title("Top 5 & Bottom 5 Predicted Stores")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nMAE: {mae:.4f}  |  R²: {r2:.4f}")

## Step 5 — Predict for a Custom Location

You can also predict the LFS for a hypothetical new location by passing your own feature values.

In [ ]:
# Example: a hypothetical Midtown location
custom_location = np.array([[
    15,        # pluto_numfloors
    1990,      # pluto_yearbuilt
    6000,      # pluto_retailarea (sqft)
    25000,     # pluto_comarea (sqft)
    10000,     # pluto_lotarea (sqft)
    8000000,   # pluto_assesstot ($)
    100,       # mta_dist_m
    30000,     # mta_avg_daily_ridership
    4,         # n_starbucks_500m
    2,         # n_dunkin_500m
    8,         # n_other_cafe_500m
    80,        # nearest_competitor_dist_m
    200,       # nearest_starbucks_dist_m
    10000,     # tract_population
    95000,     # tract_median_income
    0.40,      # tract_pct_walk_commute
    0.60,      # tract_pct_bachelors_plus
    5000,      # ped_count_nearest
    40.7549,   # lat (Midtown)
    -73.9840,  # lon (Midtown)
]])

pred = model.predict(custom_location)
print(f"Predicted LFS for custom Midtown location: {pred[0]:.3f}")
print(f"\nFor reference — dataset median LFS: {df_valid[TARGET].median():.3f}")
if pred[0] > df_valid[TARGET].median():
    print("→ This location scores ABOVE median — favorable positioning.")
else:
    print("→ This location scores BELOW median — challenging positioning.")

## Resources

- **Model:** [starbucks-location-fitness-model](https://www.kaggle.com/models/shiratoriseto/starbucks-location-fitness-model)
- **Dataset:** [manhattan-cafe-wars](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars)
- **Training notebook:** [LFS Predictive Model](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness-score-predictive-model)
- **GitHub:** [seto-siratori/starbucks-kaggle](https://github.com/seto-siratori/starbucks-kaggle)